# Diet and health with a time lag

- Inputs: `data/cleaned/gdd_cleaned_long.csv`, `data/cleaned/who_gho_clean.csv`, `data/reference/countries.csv`
- Outputs: `data/cleaned/tab_diet_health_lag.csv` (one row per country x diet year x lag x food x outcome) and `data/cleaned/tab_correlations.csv` (one row per food x outcome x lag x diet year)
- A diet year is paired with the health outcome measured `lag_years` later (5 or 10). A 2005 diet with a 10-year lag is matched to the 2015 outcome.
- GDD intakes are modelled estimates for the whole adult population (both sexes, all ages, all residences and education levels), so there are no strata to filter.
- These are country-level associations. They do not show that a food causes an outcome, and wealth affects both diet and health.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from check_schema import check_schema

REF = Path('../data/reference')
CLEANED = Path('../data/cleaned')

gdd = pd.read_csv(CLEANED / 'gdd_cleaned_long.csv')
who = pd.read_csv(CLEANED / 'who_gho_clean.csv')
countries = pd.read_csv(REF / 'countries.csv')

Step 1. Choose the foods, the lags and the outcomes. The food list follows the diet risk factors in Afshin et al. (2019), split into foods expected to protect and foods expected to raise risk. Do not change the codes or labels once the Tableau template uses them.

In [2]:
SHORTLIST = {
    # protective
    'v01': 'Protective',  # Fruits
    'v02': 'Protective',  # Non-starchy vegetables
    'v05': 'Protective',  # Beans and legumes
    'v06': 'Protective',  # Nuts and seeds
    'v08': 'Protective',  # Whole grains
    'v11': 'Protective',  # Total seafoods
    'v34': 'Protective',  # Dietary fiber
    # risk
    'v07': 'Risk',        # Refined grains
    'v09': 'Risk',        # Total processed meats
    'v10': 'Risk',        # Unprocessed red meats
    'v15': 'Risk',        # Sugar-sweetened beverages
    'v27': 'Risk',        # Saturated fat
    'v35': 'Risk',        # Added sugars
    'v37': 'Risk',        # Dietary sodium
}

LAGS = [5, 10]

# WHO column in who_gho_clean.csv -> label shown in Tableau
OUTCOMES = {
    'obesity_pct':            'Obesity (% of adults)',
    'diabetes_pct':           'Diabetes (% of adults)',
    'premature_ncd_mort_pct': 'Premature NCD death (% risk, age 30-69)',
}

# The GDD years used by the rest of the project (gdd_country_year.csv has the same seven)
DIET_YEARS = [1990, 1995, 2000, 2005, 2010, 2015, 2018]

factors = gdd[['factor_code', 'dietary_factor', 'unit']].drop_duplicates()
factors[factors['factor_code'].isin(SHORTLIST)].sort_values('factor_code')

,factor_code,dietary_factor,unit
0,v01,Fruits,grams per day
1,v02,Non-starchy vegetables,grams per day
4,v05,Beans and legumes,grams per day
5,v06,Nuts and seeds,grams per day
6,v07,Refined grains,grams per day
7,v08,Whole grains,grams per day
8,v09,Total processed meats,grams per day
9,v10,Unprocessed red meats,grams per day
10,v11,Total seafoods,grams per day
14,v15,Sugar-sweetened beverages,grams per day


Step 2. Take the diet side. One row per country x year x food, with the estimate and its 95% interval.

`gdd_cleaned_long.csv` also holds a 2020 row for four foods (v10, v33, v35, v50). They are left out here so every food has the same seven years.

In [3]:
diet = gdd[gdd['factor_code'].isin(SHORTLIST) & gdd['year'].isin(DIET_YEARS)].copy()

assert diet['factor_code'].nunique() == len(SHORTLIST), 'a shortlisted code is missing from GDD'
assert not diet.duplicated(['iso3', 'year', 'factor_code']).any(), 'a stratum is still in the data'

diet = diet.rename(columns={'year': 'diet_year', 'intake_estimate': 'intake'})
diet = diet[['iso3', 'diet_year', 'factor_code', 'dietary_factor', 'unit', 'intake']]
diet['food_group'] = diet['factor_code'].map(SHORTLIST)
diet.shape

(18130, 7)

Step 3. Pair each diet year with the outcome `lag` years later.

The outcome year is shifted back by the lag, so it lines up with the diet year it belongs to. Rows with no outcome are dropped, which is why not every diet year exists for every lag (for example a 2018 diet with a 5-year lag has no diabetes value, because the diabetes series ends in 2022).

In [4]:
frames = []
for lag in LAGS:
    for col, label in OUTCOMES.items():
        o = who[['iso3', 'year', col]].dropna(subset=[col]).copy()
        o['diet_year'] = o['year'] - lag
        o = o.rename(columns={'year': 'outcome_year', col: 'outcome_value'})
        o['lag_years'] = lag
        o['outcome_code'] = col
        o['outcome'] = label
        frames.append(diet.merge(o, on=['iso3', 'diet_year'], how='inner', validate='many_to_one'))

lag_df = pd.concat(frames, ignore_index=True)
lag_df.shape

(87276, 12)

Step 4. Add country names and regions, order the columns, check and save. TWN is in GDD but not in WHO, so it drops out here.

In [5]:
lag_df = lag_df.merge(countries.rename(columns={'who_region': 'region'}), on='iso3',
                      how='left', validate='many_to_one')
assert lag_df['region'].notna().all(), lag_df.loc[lag_df['region'].isna(), 'iso3'].unique()

lag_df = lag_df[['iso3', 'country_name', 'region', 'diet_year', 'lag_years', 'outcome_year',
                 'factor_code', 'dietary_factor', 'food_group', 'unit', 'intake',
                 'outcome_code', 'outcome', 'outcome_value']]
lag_df = lag_df.sort_values(['iso3', 'diet_year', 'lag_years', 'factor_code', 'outcome_code']).reset_index(drop=True)

check_schema(lag_df, 'tab_diet_health_lag',
             grain=('iso3', 'diet_year', 'lag_years', 'factor_code', 'outcome_code'))
lag_df.to_csv(CLEANED / 'tab_diet_health_lag.csv', index=False)

tab_diet_health_lag: 87276 rows x 14 cols | iso3: 184 | years: n/a
  OK


Step 5. Spot-check one country by hand. Singapore's 2005 fruit intake should sit next to its 2015 obesity value when the lag is 10 years.

In [6]:
row = lag_df.query("iso3 == 'SGP' and diet_year == 2005 and lag_years == 10 "
                   "and outcome_code == 'obesity_pct' and factor_code == 'v01'")
assert len(row) == 1

expected_outcome = who.query("iso3 == 'SGP' and year == 2015")['obesity_pct'].iloc[0]
expected_intake = gdd.query("iso3 == 'SGP' and year == 2005 and factor_code == 'v01'")['intake_estimate'].iloc[0]

assert row['outcome_value'].iloc[0] == expected_outcome
assert row['intake'].iloc[0] == expected_intake
row[['iso3', 'diet_year', 'outcome_year', 'dietary_factor', 'intake', 'outcome', 'outcome_value']]

,iso3,diet_year,outcome_year,dietary_factor,intake,outcome,outcome_value
68993,SGP,2005,2015,Fruits,192.07299,Obesity (% of adults),10.07


Which diet years exist for each outcome and lag. Use this to tell users why a chart can be empty.

In [7]:
lag_df.groupby(['outcome', 'lag_years'])['diet_year'].apply(lambda s: sorted(s.unique()))

outcome                                  lag_years
Diabetes (% of adults)                   5                  [1990, 1995, 2000, 2005, 2010, 2015]
                                         10                       [1990, 1995, 2000, 2005, 2010]
Obesity (% of adults)                    5            [1990, 1995, 2000, 2005, 2010, 2015, 2018]
                                         10                       [1990, 1995, 2000, 2005, 2010]
Premature NCD death (% risk, age 30-69)  5                  [1995, 2000, 2005, 2010, 2015, 2018]
                                         10                       [1990, 1995, 2000, 2005, 2010]
Name: diet_year, dtype: object

Step 6. Spearman correlation for every food x outcome x lag x diet year.

Each diet year is done separately so a country is counted once per correlation. Spearman uses ranks, so one extreme country does not drive the result. Combinations with fewer than 30 countries are skipped. The calculation is done here, not in Tableau.

In [11]:
keys = ['factor_code', 'dietary_factor', 'food_group', 'outcome_code', 'outcome', 'lag_years', 'diet_year']

rows = []
for k, g in lag_df.groupby(keys):
    n = g['iso3'].nunique()
    if n < 30:
        continue
    ranked = g[['intake', 'outcome_value']].rank()
    rho = ranked['intake'].corr(ranked['outcome_value'])
    rows.append({**dict(zip(keys, k)), 'spearman_rho': round(rho, 3), 'n_countries': n})

corr = pd.DataFrame(rows).sort_values(keys).reset_index(drop=True)

check_schema(corr, 'tab_correlations',
             grain=('factor_code', 'outcome_code', 'lag_years', 'diet_year'))
corr.to_csv(CLEANED / 'tab_correlations.csv', index=False)

tab_correlations: 476 rows x 9 cols | iso3: n/a | years: n/a
  OK


Sanity check: all foods for obesity, 2005 diet, 10-year lag, weakest to strongest.

In [12]:
(corr.query("outcome_code == 'obesity_pct' and lag_years == 10 and diet_year == 2005")
     .sort_values('spearman_rho')[['dietary_factor', 'food_group', 'spearman_rho', 'n_countries']])

,dietary_factor,food_group,spearman_rho,n_countries
191,Whole grains,Protective,-0.266,184
55,Non-starchy vegetables,Protective,-0.050,184
293,Total seafoods,Protective,-0.040,184
123,Nuts and seeds,Protective,0.009,184
157,Refined grains,Risk,0.019,184
395,Dietary fiber,Protective,0.041,184
89,Beans and legumes,Protective,0.150,184
361,Saturated fat,Risk,0.180,184
463,Dietary sodium,Risk,0.243,184
21,Fruits,Protective,0.360,184
